#Week1
#### AI Agents: Tool USE + ReAct + Memory + Multi-Agent Systems

##Pre-class Setup

In [2]:
#Same groq API Key from Week 7 works all week
#Test this loads before Monday:
from google.colab import userdata
#Install if needed: pip install groq
try:
  from groq import Groq
except ImportError:
  import subprocess; subprocess.run(["pip","install","groq","-q"])
  from groq import Groq

GROQ_API_KEY = userdata.get("kaustubh_aiml")

client = Groq(api_key=GROQ_API_KEY)
model = "openai/gpt-oss-120b"

response = client.chat.completions.create(
    model = model,
    messages=[
        {"role":"user","content":"Say Hello"}
    ]
)

###Day 1

####Define your First tool

In [ ]:
import json

#STEP 1: Write actual python function that does real work
def calculate(expression):
  """Safely evaluate a math expression and return the result"""
  try:
    result = eval(expression,{"__builtins__":{}},{})
    return {"result":result}
  except Exception as e:
    return {"error":str(e)}

def get_weather(city):
  """Simulated weather lookup (in real life, this would call a weather API)"""
  fake_weather_db = {
      "paris" : {"temp_c" : 18, "condition" : "Cloudy"},
      "tokyo" : {"temp_c" : 24, "condition" : "Sunny"},
      "mumbai" :{"temp_c" : 31, "condition": "Humid"}
  }
  city_lower = city.lower()
  if city_lower in fake_weather_db:
    return fake_weather_db[city_lower]
  return {"error": f"No weather data available for {city}"}

#Test the tools directly first --  confirm they work BEFORE giving to the LLM
print("Testing calculate():", calculate("847 * 392"))
print("Testing get_weather():", get_weather("Tokyo"))

Testing calculate(): {'result': 332024}
Testing get_weather(): {'temp_c': 24, 'condition': 'Sunny'}


###Describe those tools

In [ ]:
#Step 2: Describe each tool in the JSON Schema format the LLM understands
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluates a mathematical expression and returns"
            "the precise result. Use this for ANY arithmetic,"
            "even simple calculations, to ensure accuracy.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A math expression e.g. '847 * 392'"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Gets the current weather conditions for a specific city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name, e.g 'paris'"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

#Step 3: Send a request WITH the tools list -- let the LLM decide what to use
response = client.chat.completions.create(
    model = model,
    messages = [{"role":"user","content":"What is 847 multiplied by 392"}],
    tools = tools,
    tool_choice="auto"
)

message = response.choices[0].message
print("Did the model want to call a tool?", message.tool_calls is not None)

if message.tool_calls:
  for tool_call in message.tool_calls:
    print(f"\nTool the model wants to call: {tool_call.function.name}")
    print(f"With these arguments:  {tool_call.function.arguments}")

Did the model want to call a tool? True

Tool the model wants to call: calculate
With these arguments:  {"expression":"847 * 392"}


###Execute the Tool + Send Result Back

In [ ]:
#Step 4: Actually EXECUTE the requested tool with model's arguments
def execute_tool_call(tool_call):
  function_name = tool_call.function.name
  arguments = json.loads(tool_call.function.arguments)

  if function_name == "calculate":
    return calculate(**arguments)
  elif function_name == "get_weather":
    return get_weather(**arguments)
  else:
    return {"error": f"Unknown tool : {function_name}"}

#Run the full request-exectute-respond cycle
user_question = "What is 847 multiplied by 392"

response = client.chat.completions.create(
    model = model,
    messages = [{"role":"user","content":user_question}],
    tools = tools,
    tool_choice="auto"
)

message = response.choices[0].message

if message.tool_calls:
  tool_call = message.tool_calls[0]
  tool_result = execute_tool_call(tool_call)
  print(f"Tool executed! Real result: {tool_result}")

  #Step 5: Send the TOOL's result back to the LLM to form a final answer
  follow_up = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": user_question
        },
        message,
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(tool_result)
        }
    ]
)

final_answer = follow_up.choices[0].message.content

print(f"\nFinal Answer: {final_answer}")

Tool executed! Real result: {'result': 332024}

Final Answer: 847 multiplied by 392 equals **332,024**.


###Complete Reusable AI Loop

In [ ]:
def run_agent(user_question, tools, available_functions, model="openai/gpt-oss-120b"):
    """A complete, reusable agent loop -- THINK, ACT, OBSERVE, RESPOND"""

    messages = [
        {
            "role": "user",
            "content": user_question
        }
    ]

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    message = response.choices[0].message

    # If no tool was needed, just return the direct answer
    if not message.tool_calls:
        return message.content

    # The agent wants to ACT -- execute the tool it requested
    messages.append(message)

    for tool_call in message.tool_calls:
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        result = available_functions[function_name](**arguments)

        print(f"[Agent Action]: Called {function_name}({arguments}) ---> {result}")

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result)
        })

    # Get the FINAL response, now that the agent has real tool results
    final_response = client.chat.completions.create(
        model=model,
        messages=messages
    )

    return final_response.choices[0].message.content


available_functions = {
    "calculate": calculate,
    "get_weather": get_weather
}

test_questions = [
    "What is the weather like in Mumbai right now?",
    "If I have 24 boxes with 18 items each, how many items total?",
    "What is the capital of France?"
]

print("TESTING THE COMPLETE AGENT LOOP:\n")

for question in test_questions:
    print(f"Question: {question}")

    answer = run_agent(
        question,
        tools,
        available_functions,
        model
    )

    print(f"Answer: {answer}\n")

TESTING THE COMPLETE AGENT LOOP:

Question: What is the weather like in Mumbai right now?
[Agent Action]: Called get_weather({'city': 'Mumbai'}) ---> {'temp_c': 31, 'condition': 'Humid'}
Answer: Currently in Mumbai, it’s warm and humid with a temperature around **31 °C**.

Question: If I have 24 boxes with 18 items each, how many items total?
[Agent Action]: Called calculate({'expression': '24 * 18'}) ---> {'result': 432}
Answer: You’d have **432 items** in total (24 × 18 = 432).

Question: What is the capital of France?
Answer: The capital of France is **Paris**.



###Day 2

####The ReAct Pattern --  Reasoning and Acting in Multi-step Tasks

###Multi-step Agent Loop

In [ ]:
import json

def get_weather(city):
  """Simulated weather lookup (in real life, this would call a weather api)"""
  fake_weather_db = {
      "paris": {"temp_c":18, "conditions": "Cloudy"},
      "tokyo": {"temp_c":24, "conditions": "Sunny"},
      "mumbai": {"temp_c":31, "conditions": "Humid"}
  }
  return fake_weather_db.get(city.lower(),{"error": f"No weather data is available for {city}"})


def celsius_to_fahrenheit(celsius):
  return {"fahrenheit": round(celsius * 9/5 + 32, 1)}


tools = [
    {
        "type":"function",
        "function":{
            "name": "get_weather",
            "description": "Gets the current weather conditions for a"
                          "specific city",
            "parameters": {
                "type": "object",
                "properties" : {
                    "city" : {
                        "type": "string",
                        "description": "The city name, e.g. 'paris"
                    }
                },
                "required": ["city"]
            }
        }
    },
        {
        "type":"function",
        "function":{
            "name": "celsius_to_fahrenheit",
            "description": "Converts celsius_to_fahrenheit",
            "parameters": {
                "type": "object",
                "properties" : {
                    "celsius" : {
                        "type": "number",
                    }
                },
                "required": ["celsius"]
            }
        }
    },
]


available_functions = {
    "get_weather": get_weather,
    "celsius_to_fahrenheit": celsius_to_fahrenheit
}


def run_multistep_agent(user_question, max_steps = 5):
    """An agent that can loop through MULTIPLE tool calls before answering"""

    messages = [
        {
            "role": "user",
            "content": user_question
        }
    ]

    for step in range(max_steps):
      response = client.chat.completions.create(
          model=model,
          messages=messages,
          tools=tools,
          tool_choice="auto"
      )

      message = response.choices[0].message

      # If no tool was needed, just return the direct answer
      if not message.tool_calls:
        print(f"[Step {step+1}] Agent has enough info -- giving final answer")
        return message.content

      # The agent wants to ACT -- execute the tool it requested
      messages.append(message)
      for tool_call in message.tool_calls:
          function_name = tool_call.function.name
          arguments = json.loads(tool_call.function.arguments)

          result = available_functions[function_name](**arguments)

          print(f"[Agent Action]: Called {function_name}({arguments}) -> {result}")

          messages.append({
              "role": "tool",
              "tool_call_id": tool_call.id,
              "content": json.dumps(result)
          })

    return "Agent reached maximun steps without a final answer"

question = "What's the weather in Tokyo, and what is that temperature in Fahrenheit."
print(f"Question: {question}\n")
answer = run_multistep_agent(question)
print(f"\nFinal Answer: {answer}")


Question: What's the weather in Tokyo, and what is that temperature in Fahrenheit.

[Agent Action]: Called get_weather({'city': 'Tokyo'}) -> {'temp_c': 24, 'conditions': 'Sunny'}
[Agent Action]: Called celsius_to_fahrenheit({'celsius': 24}) -> {'fahrenheit': 75.2}
[Step 3] Agent has enough info -- giving final answer

Final Answer: The current weather in **Tokyo** is **sunny** with a temperature of **24 °C**.  

Converted to Fahrenheit, that’s **75.2 °F**.


####Sequential Tool Chaining, Visualised

In [ ]:
def run_multistep_agent_verbose(user_question, max_steps = 5):
    """Same agent loop, but tracking the full THOUGHT/ACTION/OBSERVATION trace"""

    messages = [
        {
            "role": "user",
            "content": user_question
        }
    ]
    trace = []

    for step in range(max_steps):
      response = client.chat.completions.create(
          model=model,
          messages=messages,
          tools=tools,
          tool_choice="auto"
      )

      message = response.choices[0].message

      # If no tool was needed, just return the direct answer
      if not message.tool_calls:
        trace.append({"step": step+1, "type":"FINAL ANSWER", "content": message.content})
        break

      # The agent wants to ACT -- execute the tool it requested
      messages.append(message)

      for tool_call in message.tool_calls:
          function_name = tool_call.function.name
          arguments = json.loads(tool_call.function.arguments)

          result = available_functions[function_name](**arguments)

          trace.append({"step": step+1, "type": "ACTION",
                        "content": f"{function_name}({arguments}"})
          trace.append({"step": step+1, "type": "OBSERVATION",
                        "content": str(result)})

          messages.append({
              "role": "tool",
              "tool_call_id": tool_call.id,
              "content": json.dumps(result)
          })

    return trace

trace = run_multistep_agent_verbose(
    "Check whether in Paris, and tell me if I should wear a coat"
    "based on whether if its below 15 Celsius"
)

print("FULL AGENT REASONING TRACE:")
print("="* 55)
for entry in trace:
  print(f"[step {entry['step']}]  {entry['type']}: {entry['content']}")

FULL AGENT REASONING TRACE:
[step 1]  ACTION: get_weather({'city': 'Paris'}
[step 1]  OBSERVATION: {'temp_c': 18, 'conditions': 'Cloudy'}
[step 2]  FINAL ANSWER: The current temperature in Paris is **18 °C**, which is above the 15 °C threshold. You should be comfortable without a coat. Enjoy your day!


####A Research-Style Agent

In [ ]:
import json

# --- 1. Define Research Functions ---
def search_company_info(company_name):
    """Get basic company information (industry, employee count, founding year)"""
    fake_db = {
        "techcorp": {"industry": "Software", "employees": 5000, "founded": 2010},
        "retailco": {"industry": "Retail", "employees": 15000, "founded": 1995}
    }
    return fake_db.get(company_name.lower(), {"error": "Company not found"})


def get_stock_price(ticker):
    """Get the current stock price for a ticker symbol"""
    fake_prices = {"TECH": 145.32, "RETL": 67.89}
    return {"ticker": ticker, "price": fake_prices.get(ticker.upper(), None)}


def calculate_market_cap(price, shares_outstanding):
    """Calculate market capitalization given price and shares outstanding"""
    return {"market_cap_millions": round(price * shares_outstanding / 1_000_000, 2)}


# --- 2. Define JSON Schemas for Tools ---
research_tools = [
    {
        "type": "function",
        "function": {
            "name": "search_company_info",
            "description": "Get basic company information (industry, employee count, founding year)",
            "parameters": {
                "type": "object",
                "properties": {
                    "company_name": {
                        "type": "string",
                        "description": "The name of the company, e.g. 'TechCorp'"
                    }
                },
                "required": ["company_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "Get the current stock price for a ticker symbol",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {
                        "type": "string",
                        "description": "The stock ticker symbol, e.g. 'TECH'"
                    }
                },
                "required": ["ticker"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_market_cap",
            "description": "Calculate market capitalization given price and shares outstanding",
            "parameters": {
                "type": "object",
                "properties": {
                    "price": {
                        "type": "number",
                        "description": "Stock price per share"
                    },
                    "shares_outstanding": {
                        "type": "number",
                        "description": "Total number of outstanding shares"
                    }
                },
                "required": ["price", "shares_outstanding"]
            }
        }
    }
]

# --- 3. Function Mapping ---
research_functions = {
    "search_company_info": search_company_info,
    "get_stock_price": get_stock_price,
    "calculate_market_cap": calculate_market_cap
}


# --- 4. Research Agent Function ---
def run_research_agent(question, tools_list, function_dict, max_steps=6):
    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    for step in range(max_steps):
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools_list,
            tool_choice="auto"
        )

        message = response.choices[0].message

        # If the agent is done calling tools, return the final answer
        if not message.tool_calls:
            print(f"[Step {step+1}] Agent has enough info -- giving final answer")
            return message.content

        # Append assistant's tool call decision to context
        messages.append(message)

        # Execute requested tools and append observations
        for tool_call in message.tool_calls:
            fname = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)

            result = function_dict[fname](**arguments)
            print(f"[Step {step+1}] Action: {fname}({arguments}) -> {result}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })

    return "Max steps reached without a final answer"


# --- 5. Execution ---
question = "research TechCorp for me: what industry are they in and what is TECH stock price?"
print(f"Question: {question}\n")

answer = run_research_agent(question, research_tools, research_functions)
print(f"\nFinal Answer:\n{answer}")

Question: research TechCorp for me: what industry are they in and what is TECH stock price?

[Step 1] Action: search_company_info({'company_name': 'TechCorp'}) -> {'industry': 'Software', 'employees': 5000, 'founded': 2010}
[Step 2] Action: get_stock_price({'ticker': 'TECH'}) -> {'ticker': 'TECH', 'price': 145.32}
[Step 3] Agent has enough info -- giving final answer

Final Answer:
**TechCorp Overview**

- **Industry:** Software  
- **Founded:** 2010  
- **Employees:** Approximately 5,000  

**Current Stock Information (Ticker: TECH)**  

- **Latest Price:** **$145.32** per share  

If you need more details—such as recent earnings, market capitalization, or a deeper dive into their product offerings—just let me know!


####Handling Agent Mistakes

In [ ]:
print("WHAT HAPPENS WHEN A TOOL RETURNS AN ERROR?")
print("="*55)

#Ask about a company what doesnt exist in our fake database
question_with_errors = "What industry is 'NoneexistentCorp' in?"
print(f"Question: {question_with_errors}\n")
answer = run_research_agent(question_with_errors, research_tools, research_functions)
print(f"\nFinal Answer: {answer}")

print(f"""
NOTICE: When the tool returned an error (company not found), the
agent didn't crash -- it INCORPORATED that error information into
its reasoning and gave an honest, helpful response acknowledging the limitation.

THIS IS A CRITICAL DESIGN PRINCIPLE:

Tools should return CLEAR, INFORMATIVE error messages (not just
crash or return None). The agent can then reason about the error
and respond appropriately, rather than getting confused or hallucinating a fake answer.

BAD tool design: return None on failure (agent might hallucinate!)
GOOD tool design: return {{'error': 'specific reason'}} (agent can
                explain the limitation honestly to the user)
""")


WHAT HAPPENS WHEN A TOOL RETURNS AN ERROR?
Question: What industry is 'NoneexistentCorp' in?

[Step 1] Action: search_company_info({'company_name': 'NoneexistentCorp'}) -> {'error': 'Company not found'}
[Step 2] Agent has enough info -- giving final answer

Final Answer: I’m sorry, but I couldn’t find any information about a company named **NoneexistentCorp**, so I’m unable to determine its industry. If you have a different company in mind or can provide more details, feel free to let me know!

NOTICE: When the tool returned an error (company not found), the
agent didn't crash -- it INCORPORATED that error information into
its reasoning and gave an honest, helpful response acknowledging the limitation.

THIS IS A CRITICAL DESIGN PRINCIPLE:

Tools should return CLEAR, INFORMATIVE error messages (not just
crash or return None). The agent can then reason about the error
and respond appropriately, rather than getting confused or hallucinating a fake answer.

BAD tool design: return None on

###Day 3

####Giving agents Memory - Building Stateful, Context-Aware Agents

####Conversational Memory in Agent

In [ ]:
class ConversationalAgent:
  """An Agent that remembers the CURRENT conversation"""
  def __init__(self, system_prompt="You are a helpful assistant."):
    self.history = [{"role":"user","content":system_prompt}]

  def chat(self, user_message):
    self.history.append({"role":"user","content":user_message})
    response = client.chat.completions.create(
        model=model, messages=self.history
    )
    assistant_reply = response.choices[0].message.content
    self.history.append({"role":"assistant","content":assistant_reply})
    return assistant_reply

agent = ConversationalAgent(
    system_prompt = "You are a helpful data science tutor."
)

print("Turn 1:")
print(agent.chat("My name is Alex and I'm learning about neural networks."))

print("\nTurn 2:")
print(agent.chat("What's a good first project for me to try?"))

print("\nTurn 3(testing memory):")
print(agent.chat("What's my name, and what am I currently learning about?"))

print(f"Conversation now has {len(agent.history)} messages stored.")
print(f"This Is the agent's 'memory' for this session == it's simply")
print(f"this full message history, resent with every new turn.")

Turn 1:
Hi Alex! 👋 Great to meet you, and welcome to the fascinating world of neural networks. I’m happy to help you explore the concepts, math, and practical tools behind them.

### Quick Overview of What a Neural Network Is

| Concept | What It Means | Why It Matters |
|---------|---------------|----------------|
| **Neuron (or node)** | A simple mathematical function that takes inputs, applies weights, adds a bias, and passes the result through an activation function. | The building block of any network—just like biological neurons, they “fire” based on the input they receive. |
| **Layer** | A collection of neurons that operate in parallel. | Stacking layers lets the model learn increasingly abstract features. |
| **Weights & Biases** | Parameters the model learns during training. | Adjusting these lets the network fit the data. |
| **Activation Function** | Non‑linear transformation (e.g., ReLU, sigmoid, tanh). | Enables the network to model complex, non‑linear relationships. |
| 

####Structured User Profile Memory

In [ ]:

import json

class SmartMemoryAgent:
    """An agent that extracts and remembers KEY FACTS, not just raw conversations."""

    def __init__(self, system_prompt="You are a helpful assistant."):
        self.system_prompt = system_prompt
        self.history = [{"role": "system", "content": system_prompt}]
        self.user_profile = {}  # Structured facts about the user

    def extract_facts(self, user_message):
        """Use the LLM itself to extract structured facts worth remembering."""
        extraction_prompt = f"""Extract any PERSONAL FACTS worth remembering long-term from this message (name, preferences, goals, skill level, etc.).
Respond ONLY with JSON formatted like: {{"facts": {{"key": "value"}}}}
If there is nothing worth remembering, respond with: {{"facts": {{}}}}

User Message: "{user_message}"
"""
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": extraction_prompt}],
            response_format={"type": "json_object"}
        )

        try:
            parsed = json.loads(response.choices[0].message.content)
            return parsed.get("facts", {})
        except Exception as e:
            print(f"Fact extraction failed: {e}")
            return {}

    def chat(self, user_message):
        # 1. Extract and store any new facts BEFORE responding
        new_facts = self.extract_facts(user_message)
        self.user_profile.update(new_facts)

        #Build a system message that includes what we know about the user
        profile_context = ""
        if self.user_profile:
          profile_context = f"\n\nKnown facts about this user: {json.dumps(self.user_profile)}"

        messages = [
            {"role": "system","content":self.history[0]["content"] + profile_context},
            {"role":"user", "content": user_message}
        ]
        response = client.chat.completions.create(model=model, messages=messages)
        return response.choices[0].message.content

smart_agent = SmartMemoryAgent(system_prompt="You are a helpful data science tutor.")

print(smart_agent.chat("Hi, I'm Luffy, I work in finance and I'm a complete beginner at Python."))
print(f"\nExtracted profile so far: {smart_agent.user_profile}")

print("\n" + smart_agent.chat("Can you suggest  a good learning project for me?"))
print(f"\nExtracted profile so far: {smart_agent.user_profile}")


Hi Luffy! 👋  
Great to meet you. I’m happy to help you get started with Python and guide you toward using it for finance‑focused data work.

### Quick Roadmap for a Python Beginner in Finance

| Step | What to Learn | Why it matters for finance | Suggested Resources |
|------|---------------|----------------------------|----------------------|
| **1️⃣ Set up your environment** | Install Python (via Anaconda) and a code editor (VS Code or JupyterLab). | Gives you a stable, reproducible workspace for data analysis. | - <https://www.anaconda.com/products/distribution>  <br> - Intro video: “Getting Started with Jupyter Notebook” (YouTube, 10 min) |
| **2️⃣ Python basics** | Variables, data types, control flow (if/for/while), functions, basic I/O. | These are the building blocks for any script you’ll write—e.g., pulling data, cleaning it, running calculations. | - “Python for Everybody” (Free Coursera) – first 3 weeks <br> - “Automate the Boring Stuff with Python” (Chapters 1‑4) – free onli

####Persistent Memory Across Sessions

In [ ]:

import json
import os

class PersistentMemoryAgent:
    """An agent whose memory SURVIVES even after the program restarts"""

    def __init__(self, memory_file="user_memory.json", system_prompt="You are a helpful assistant."):
        self.memory_file = memory_file
        self.system_prompt = system_prompt
        self.user_profile = self._load_memory()
        self.history = [{"role": "system", "content": self.system_prompt}]

    def _load_memory(self):
        """Load previously saved facts from disk, if they exist"""
        if os.path.exists(self.memory_file):
            try:
                with open(self.memory_file, "r") as f:
                    return json.load(f)
            except Exception as e:
                print(f"Error loading memory: {e}")
                return {}
        return {}

    def _save_memory(self):
        """Persist current facts to disk"""
        try:
            with open(self.memory_file, "w") as f:
                json.dump(self.user_profile, f, indent=2)
        except Exception as e:
            print(f"Error saving memory: {e}")

    def extract_facts(self, user_message):
        """Use the LLM itself to extract structured facts worth remembering."""
        extraction_prompt = f"""Extract PERSONAL FACTS worth remembering long-term (name, preferences, goals, skill level) from this message.

Respond ONLY with JSON:
{{"facts": {{"key": "value"}}}}

If there's nothing worth remembering, respond ONLY with:
{{"facts": {{}}}}

Message: "{user_message}"
"""
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": extraction_prompt}],
            temperature=0
        )

        try:
            cleaned = response.choices[0].message.content.strip()
            cleaned = cleaned.strip("```json\n").strip("```").strip()
            extracted = json.loads(cleaned)
            return extracted.get("facts", {})
        except Exception as e:
            print(f"Fact extraction failed: {e}")
            return {}

    def chat(self, user_message):
        # 1. Extract and store any new facts BEFORE responding
        new_facts = self.extract_facts(user_message)
        if new_facts:
            self.user_profile.update(new_facts)
            self._save_memory()  # Save to disk!
            print(f"[Memory Persisted to Disk]: {new_facts}")

        # 2. Build system message with persistent user profile
        profile_context = ""
        if self.user_profile:
            profile_context = f"\nKnown facts about the user: {json.dumps(self.user_profile)}"

        messages = [
            {
                "role": "system",
                "content": self.history[0]["content"] + profile_context
            },
            {
                "role": "user",
                "content": user_message
            }
        ]

        # 3. Request answer from LLM
        response = client.chat.completions.create(
            model=model,
            messages=messages
        )

        return response.choices[0].message.content

#SESSION 1: -- SIMULATE A USER's first conversation
print("--- SESSION 1---")
agent_session1 = PersistentMemoryAgent(memory_file="demo_memoy.json")
print(agent_session1.chat("Hi, I'm Zoro and I'm studying for a data science interview."))
print(f"Memory saved to disk: {agent_session1.user_profile}")

#SESSION 1: -- SIMULATED  restarting the program completely (brand new agent object!)
print("\n--- SESSION 2 (completely new agent object, simulating a restart) ---")
agent_session2 = PersistentMemoryAgent(memory_file="demo_memory.json")
print(f"Memory loaded from disk:  {agent_session2.user_profile}")
print(agent_session2.chat("What was I preparing for again?"))

--- SESSION 1---
[Memory Persisted to Disk]: {'name': 'Zoro', 'goal': 'studying for a data science interview'}
Hey Zoro! 👋 Great to meet you, and good luck with your data‑science interview prep. Below is a concise roadmap plus some practical resources to help you hit the high‑notes across the typical interview stages.

---

## 1️⃣ Interview Flow Overview

| Stage | What’s Usually Tested | How to Prepare |
|-------|----------------------|----------------|
| **Screening / HR** | Fit, motivation, basic background | Prepare a 2‑minute “elevator pitch” that links your past projects to the role. |
| **Technical Phone/Video** | Coding (Python/R/SQL), stats basics, product sense | 1–2 coding problems + 1‑2 statistics questions; keep a timer. |
| **On‑site / Virtual Deep‑Dive** | • **Coding** (data‑wrangling, algorithms) <br>• **Statistics / ML** (model selection, evaluation) <br>• **Case Study / Business Problem** <br>• **Product & Communication** | Structured practice (see sections below). |


#### Memory-Aware Agent With Tools Combined

In [ ]:
import json, os

# --- Reuse PersistentMemoryAgent exactly as-is ---
class PersistentMemoryAgent:
    def __init__(self, memory_file="user_memory.json",
                system_prompt="You are a helpful assistant."):
        self.memory_file = memory_file
        self.system_prompt = system_prompt
        self.user_profile = self._load_memory()

    def _load_memory(self):
        if os.path.exists(self.memory_file):
            with open(self.memory_file, "r") as f:
                return json.load(f)
        return {}

    def _save_memory(self):
        with open(self.memory_file, "w") as f:
            json.dump(self.user_profile, f, indent=2)

    def extract_facts(self, user_message):
        prompt = f"""Extract PERSONAL FACTS worth remembering long-term
(name, preferences, goals, skill level) from this message.
Respond ONLY with JSON: {{"facts": {{"key": "value"}}}}
If nothing worth remembering, respond: {{"facts": {{}}}}
Message: "{user_message}" """
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        try:
            cleaned = response.choices[0].message.content.strip().strip("```json").strip("```")
            return json.loads(cleaned).get("facts", {})
        except:
            return {}

# --- The tool from Day 3's illustrative cell, now genuinely wired in ---
def get_study_resources(topic, skill_level):
    resources = {
        ("neural networks", "beginner"): "Start with 3Blue1Brown's neural network video series",
        ("neural networks", "intermediate"): "Try implementing backpropagation from scratch",
        ("python", "beginner"): "Codecademy's Python course is a great starting point",
    }
    key = (topic.lower(), skill_level.lower())
    return {"recommendation": resources.get(key, f"General resources for {topic}")}

tools = [{
    "type": "function",
    "function": {
        "name": "get_study_resources",
        "description": "Get a personalized learning resource recommendation "
                       "for a given topic, tailored to the user's skill level.",
        "parameters": {
            "type": "object",
            "properties": {
                "topic": {"type": "string", "description": "The subject to learn"},
                "skill_level": {"type": "string",
                                "description": "The user's current skill level "
                                              "(beginner, intermediate, advanced)"}
            },
            "required": ["topic", "skill_level"]
        }
    }
}]
available_functions = {"get_study_resources": get_study_resources}

# --- a memory-aware agent loop ---
class MemoryAwareToolAgent(PersistentMemoryAgent):
    def chat(self, user_message):
        # STEP 1: extract + save any new facts, exactly like Day 3
        new_facts = self.extract_facts(user_message)
        if new_facts:
            self.user_profile.update(new_facts)
            self._save_memory()

        # STEP 2: inject remembered facts directly into the system prompt,
        # so the LLM KNOWS them without the user repeating anything
        profile_context = f"\n\nKnown facts about this user: {json.dumps(self.user_profile)}" \
                          if self.user_profile else ""

        messages = [
            {"role": "system", "content": self.system_prompt + profile_context},
            {"role": "user", "content": user_message}
        ]

        # STEP 3: normal agent tool-calling loop (Day 1-2's pattern)
        response = client.chat.completions.create(
            model=model, messages=messages,
            tools=tools, tool_choice="auto"
        )
        message = response.choices[0].message

        if not message.tool_calls:
            return message.content

        messages.append(message)
        for tool_call in message.tool_calls:
            fname = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            print(f"  [Agent called]: {fname}({args})")   # watch what gets passed in!
            result = available_functions[fname](**args)
            messages.append({"role": "tool", "tool_call_id": tool_call.id,
                            "content": json.dumps(result)})

        final = client.chat.completions.create(model=model, messages=messages)
        return final.choices[0].message.content

agent = MemoryAwareToolAgent(memory_file="study_demo.json",
                             system_prompt="You are a helpful study assistant.")

print("--- Turn 1: user reveals their skill level ---")
print(agent.chat("Hi, I'm a complete beginner just starting to learn about neural networks."))
print(f"\nRemembered profile: {agent.user_profile}")

print("\n--- Turn 2: user asks for resources, WITHOUT restating skill level ---")
print(agent.chat("Can you recommend some learning resources for neural networks?"))

print("""
IMAGINE: an agent that remembers "Marcus is a beginner" from Day 3's
memory system, and AUTOMATICALLY passes that skill_level into a
tool call like get_study_resources(topic="neural networks",
skill_level="beginner") -- without the user needing to repeat their
skill level every single time they ask a question.

This is the genuine payoff of combining Day 1-2's TOOLS with Day 3's
MEMORY: an agent that gets progressively more personalized and useful
the more it interacts with a specific user, while still being able
to take real actions via tools.
""")

--- Turn 1: user reveals their skill level ---
  [Agent called]: get_study_resources({'skill_level': 'beginner', 'topic': 'neural networks'})
Welcome to the world of neural networks! 🎉  
Since you’re just starting out, I’ll walk you through the basic ideas, give you a simple example you can try yourself, and point you to beginner‑friendly resources to keep the momentum going.

---

## 1️⃣ What Is a Neural Network (in Plain English)?

Think of a neural network as a **tiny, simplified model of how the brain works**:

| Brain | Neural Network |
|-------|----------------|
| **Neurons** receive signals, process them, and pass them on. | **Artificial neurons (or “nodes”)** take numbers as input, do a tiny calculation, and output a new number. |
| Neurons are connected by **synapses** that can strengthen or weaken. | Connections are called **weights**; they determine how much influence one node has on another. |
| The brain learns by adjusting synapse strengths based on experience. | A neural

###Day 4

####MCP (Model Context Protocol) - Standardizing the Tools You Built By Hand

####Build an MCP server With FastMCP

In [ ]:
!pip install "mcp[cli]>=1.28,<2" -q

In [ ]:
%%writefile weather_server.py
from mcp.server.fastmcp import FastMCP

#Create the MCP server -- give it a name
mcp = FastMCP("WeatherTools")

#Reuse the EXACT SAME logic from Week 8, Day 1's get_weather() function!
#The only difference: @mcp.tool() replaces manually writing JSON schema
@mcp.tool()
def get_weather(city):
  """Simulated weather lookup (in real life, this would call a weather API)"""
  fake_weather_db = {
      "paris" : {"temp_c" : 18, "condition" : "Cloudy"},
      "tokyo" : {"temp_c" : 24, "condition" : "Sunny"},
      "mumbai" :{"temp_c" : 31, "condition": "Humid"}
  }
  city_lower = city.lower()
  if city_lower in fake_weather_db:
    return fake_weather_db[city_lower]
  return {"error": f"No weather data available for {city}"}

@mcp.tool()
def calculate(expression: str) -> dict:
  """Safely evaluate a math expression and return the result"""
  try:
    result = eval(expression,{"__builtins__":{}},{})
    return {"result":result}
  except Exception as e:
    return {"error":str(e)}

if __name__ == "__main__":
  mcp.run()

Overwriting weather_server.py


In [ ]:
#Test the underlying functions directly first, exactly like Day 1's
#"test the tools directly before giving them to the LLM" step
import sys
sys.path.insert(0,".")

from weather_server import get_weather, calculate

print("Testing tools directly (bypassing MCP protocol, just calling the)")
print("underlying Pythong functions to confirm the logic works): \n")
print("get_weather('Tokyo'):", get_weather("Tokyo"))
print("calulate('847 * 392'):", calculate("847 * 392"))

Testing tools directly (bypassing MCP protocol, just calling the)
underlying Pythong functions to confirm the logic works): 

get_weather('Tokyo'): {'temp_c': 24, 'condition': 'Sunny'}
calulate('847 * 392'): {'result': 332024}


In [ ]:
!pip install "mcp[cli]>=1.28, <2" nest_asyncio -q

import nest_asyncio
nest_asyncio.apply()

import asyncio
from mcp.server.fastmcp import FastMCP

async def demo_mcp_tool_discovery():
  from weather_server import mcp as weather_mcp
  tools = await weather_mcp.list_tools()
  print("Tools discovered on this MCP server:")
  for tool in tools:
    print(f"\n Name:  {tool.name}")
    print(f"  Description:  {tool.description}")
    print(f"  Input schema: {tool.inputSchema}")

asyncio.run(demo_mcp_tool_discovery())

Tools discovered on this MCP server:

 Name:  get_weather
  Description:  Simulated weather lookup (in real life, this would call a weather API)
  Input schema: {'properties': {'city': {'title': 'city', 'type': 'string'}}, 'required': ['city'], 'title': 'get_weatherArguments', 'type': 'object'}

 Name:  calculate
  Description:  Safely evaluate a math expression and return the result
  Input schema: {'properties': {'expression': {'title': 'Expression', 'type': 'string'}}, 'required': ['expression'], 'title': 'calculateArguments', 'type': 'object'}


####End-to-End: A Full Agent Pipeline Powered Entirely by MCP

In [ ]:
%%writefile mcp_agent_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("TripPlanningTools")

@mcp.tool()
def get_weather(city: str) -> dict:
    """Get the current weather (temperature in Celsius) for a specific city."""
    fake_weather_db = {
        "tokyo": {"temp_c": 24, "condition": "Sunny"},
        "paris": {"temp_c": 12, "condition": "Rainy"},
        "mumbai": {"temp_c": 33, "condition": "Humid"},
    }
    return fake_weather_db.get(city.lower(), {"error": f"No data for {city}"})

@mcp.tool()
def celsius_to_fahrenheit(celsius: float) -> dict:
    """Convert a temperature from Celsius to Fahrenheit."""
    return {"fahrenheit": round(celsius * 9/5 + 32, 1)}

@mcp.tool()
def convert_currency(amount: float, from_currency: str, to_currency: str) -> dict:
    """Convert an amount of money from one currency to another."""
    fake_rates_to_usd = {"usd": 1.0, "eur": 1.08, "jpy": 0.0067, "inr": 0.012}
    from_rate = fake_rates_to_usd.get(from_currency.lower())
    to_rate = fake_rates_to_usd.get(to_currency.lower())
    if from_rate is None or to_rate is None:
        return {"error": "Unsupported currency"}
    usd_value = amount * from_rate
    converted = usd_value / to_rate
    return {"converted_amount": round(converted, 2), "currency": to_currency.upper()}

if __name__ == "__main__":
    mcp.run()

Writing mcp_agent_server.py


####Build MCP powered agent client

In [ ]:
import json
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


async def run_mcp_agent(user_question, max_steps=6):
    server_params = StdioServerParameters(
        command="python", args=["mcp_agent_server.py"]
    )

    # THE FIX: errlog=sys.__stderr__ instead of the Jupyter-wrapped default
    async with stdio_client(server_params, errlog=sys.__stderr__) as (
        read,
        write,
    ):
        async with ClientSession(read, write) as session:
            await session.initialize()

            mcp_tools = (await session.list_tools()).tools
            groq_tools = [mcp_tool_to_groq_format(t) for t in mcp_tools]
            print(
                f"Discovered {len(groq_tools)} tools from the MCP server: "
                f"{[t.name for t in mcp_tools]}\n"
            )

            messages = [{"role": "user", "content": user_question}]

            for step in range(max_steps):
                response = client.chat.completions.create(
                    model=model,
                    messages=messages,
                    tools=groq_tools,
                    tool_choice="auto",
                )
                message = response.choices[0].message

                if not message.tool_calls:
                    return message.content

                messages.append(message)

                for tool_call in message.tool_calls:
                    fname = tool_call.function.name
                    args = json.loads(tool_call.function.arguments)

                    mcp_result = await session.call_tool(
                        fname, arguments=args
                    )
                    result_text = mcp_result.content[0].text

                    print(
                        f"  [Step {step + 1}] MCP call: {fname}({args}) -> {result_text}"
                    )

                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": result_text,
                    })

            return "Max steps reached."


####Run it end to end:

In [ ]:
question = ("I'm travelling to Tokyo. What's the weather in Fahrenheit,"
            "and if I bring 200 USD, how much is that in Japanese Yen?")

answer = asyncio.run(run_mcp_agent(question))
print(f"\nFinal Answer: {answer}")

Discovered 3 tools from the MCP server: ['get_weather', 'celsius_to_fahrenheit', 'convert_currency']

  [Step 1] MCP call: get_weather({'city': 'Tokyo'}) -> {
  "temp_c": 24,
  "condition": "Sunny"
}
  [Step 2] MCP call: celsius_to_fahrenheit({'celsius': 24}) -> {
  "fahrenheit": 75.2
}
  [Step 3] MCP call: convert_currency({'amount': 200, 'from_currency': 'USD', 'to_currency': 'JPY'}) -> {
  "converted_amount": 29850.75,
  "currency": "JPY"
}

Final Answer: **Current Weather in Tokyo**  
- Temperature: **75.2 °F** (Sunny)

**Currency Conversion**  
- **200 USD** ≈ **29,850.75 JPY** 

Safe travels! Let me know if you need anything else—like a packing checklist or local attractions.


###Day 5

####Two specialized Agents

In [3]:
def call_agent(system_prompt, user_message, temperature=0.5):
  """A reusable function to call any single agent with its own persona"""
  response = client.chat.completions.create(
      model = model,
      messages = [
          {"role":"system","content":system_prompt},
          {"role":"user","content":user_message}
      ],
      temperature=temperature
  )
  return response.choices[0].message.content

#Agent 1: The Researcher -- gathers raw information
researcher_prompt = """You are a senior Market Researcher. You job is
to provide factual, well-organized information on a given topic.
Focus on concrete facts and figures, not opinions or recommendations."""

#Agent 2: The Writer -- turns findings into polished content
writer_prompt = """You are a Professional Content Writer. Your job is
to take raw research findings and turn them into a clear, engaging,
well-structured summary suitable for a business audience"""

topic = "The current state of the elecctric vehicle market"

print("--- AGENT 1: RESEARCHER ---")
research_findings = call_agent(researcher_prompt,
    f"Provide key facts about:  {topic}")
print(research_findings)

print("\n---AGENT 2: WRITER ---")
final_report = call_agent(writer_prompt,
    f"Turn these research findings into an engaging summary: \n\n{research_findings}")
print(final_report)

print(f"\nNOTICE: Agent 2 never researched anything itself -- it received")
print(f"Agent 1's OUTPUT as its INPUT, and focused entirely on its own.")
print(f"speciality: clear, engaging writing. This is agent HANDOFF")

--- AGENT 1: RESEARCHER ---
**The Electric‑Vehicle (EV) Market – Key Facts (as of Q2 2026)**  

| Metric | Figure | Source / Date |
|--------|--------|---------------|
| **Global EV sales (BEV + PHEV)** | **13.2 million units** in 2025 (≈ 28 % of total new‑car registrations) | International Energy Agency (IEA) – *Global EV Outlook 2026* (published Jan 2026) |
| **Cumulative EV stock** | **31.9 million** (BEV ≈ 22 M, PHEV ≈ 9.9 M) | EV‑Volumes.com – *2025 Year‑End Report* (Oct 2025) |
| **Year‑over‑year growth (2024 → 2025)** | **+38 %** in sales volume | BloombergNEF – *Electric Vehicle Outlook 2025* (Dec 2025) |
| **EV share of new‑car registrations** | **28 %** globally; **44 %** in Europe, **38 %** in China, **19 %** in the United States | IEA – *Global EV Outlook 2026* |
| **Top five markets by 2025 sales** | 1. China – 6.1 M (≈ 46 % of global) 2. Europe – 3.8 M 3. United States – 1.6 M 4. India – 0.9 M 5. Japan – 0.5 M | EV‑Volumes.com |
| **Battery‑electric vehicle (BEV) vs. plug

####A Sequential 3-Agent PipeLine

In [6]:
class Agent:
    def __init__(self, name, role_description, temperature=0.5, max_tokens=150):
        self.name = name
        self.role_description = role_description
        self.temperature = temperature
        self.max_tokens = max_tokens

    def run(self, input_text):
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": self.role_description},
                {"role": "user", "content": input_text}
            ],
            temperature=self.temperature,
            max_tokens=self.max_tokens
        )
        return response.choices[0].message.content


# --- Define the 3 Agents ---
researcher = Agent(
    "Researcher",
    "You are a market researcher. Provide factual findings on the given topic.",
    temperature=0.3
)

analyst = Agent(
    "Analyst",
    "You are a data analyst. Given research findings, identify the 3 most "
    "important insights or implications for a business.",
    temperature=0.4
)

writer = Agent(
    "Writer",
    "You are a content writer. Given analytical insights, write a short, "
    "polished executive summary (3-4 sentences) for a business audience.",
    temperature=0.6
)


# --- Pipeline Runner ---
def run_pipeline(topic, agents):
    """Run a topic through a SEQUENCE of agents, each building on the last"""
    current_output = f"Research topic: {topic}"

    for agent in agents:
        print(f"\n--- {agent.name} is working ---")
        current_output = agent.run(current_output)
        print(current_output)

    return current_output


# --- Execute Pipeline ---
topic = "The impact of remote work on employee productivity"
final_result = run_pipeline(topic, [researcher, analyst, writer])

print(f"\n{'=' * 55}")
print(f"FINAL PIPELINE OUTPUT:")
print(f"{'=' * 55}")
print(final_result)


--- Researcher is working ---
**Research Brief – The Impact of

--- Analyst is working ---
It looks like the research brief got cut off. Could you please share the full findings or the complete “Research Brief – The Impact of …” so I can review the data and pull out the three most important insights for

--- Writer is working ---
I’m sorry—I don’t have access to the full “Research Brief – The Impact of …”

FINAL PIPELINE OUTPUT:
I’m sorry—I don’t have access to the full “Research Brief – The Impact of …”


####Agent Handoff With Tools

In [11]:
import json

# combine multi agent pipelines WITH tool use from earlier in the week

# 1. Tool Function
def get_company_data(company_name):
  fake_db = {"techcorp": {"revenue_millions": 450, "employees": 5000, "growth_rate": 0.15}}
  return fake_db.get(company_name.lower(), {"error": "No data"})

# 2. Tool Definition (Fixed JSON schema syntax)
data_tools = [{
    "type": "function",
    "function": {
        "name": "get_company_data",
        "description": "Get financial data for a company",
        "parameters": {
            "type": "object",
            "properties": {
                "company_name": {
                    "type": "string",
                    "description": "The name of the company to look up"
                }
            },
            "required": ["company_name"]
        }
    }
}]

# 3. Researcher Agent
def researcher_with_tools(company_name):
  """Researcher agent that can use tools to gather REAL data"""
  messages = [
      {"role": "system", "content": "You are a financial researcher. Use tools to gather real company data."},
      {"role": "user", "content": f"Gather financial data about {company_name}'s financial data"}
  ]

  # Initial model call
  response = client.chat.completions.create(
      model=model,
      messages=messages,
      tools=data_tools,
      tool_choice="auto"
  )

  message = response.choices[0].message

  # If tool call requested, execute and send result back to model
  if message.tool_calls:
    messages.append(message)
    for tool_call in message.tool_calls:
      args = json.loads(tool_call.function.arguments)
      result = get_company_data(**args)
      messages.append({
          "role": "tool",
          "tool_call_id": tool_call.id,
          "content": json.dumps(result)
      })

    # Final response with tool results included
    follow_up = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return follow_up.choices[0].message.content

  return message.content

# Test call
print(researcher_with_tools("techcorp"))


# Step 1 RESEARCHER AGENT USES TO TOOLS TO GATHER REAL DATA



research_output = researcher_with_tools("techcorp")
print(f"\nResearch Output: {research_output}")
print("--- Researcher (with tool access)----")
print(research_output)


analyst_output = analyst.run(research_output)
print(f"\nAnalyst Output: {analyst_output}")
print("--- Analyst (no tool access)----")
print(analyst_output )


writer_output = writer.run(analyst_output)
print(f"\nWriter Output: {writer_output}")
print("--- Writer (no tool access)----")
print(writer_output)

print(f"\n{'=' * 55}")
print("This combines EVERYTHING from this week : tool use (Day 1- 2)")
print(f"{'=' * 55}")
print("and mulitple SPECIALIZED gents handing off work in sequence.")

Here is the latest financial snapshot for **TechCorp**:

| Metric | Value |
|--------|-------|
| **Revenue** | **$450 million** |
| **Number of Employees** | **5,000** |
| **Growth Rate** | **15 % YoY** |

These figures give a quick view of TechCorp’s scale and recent performance. If you need additional details—such as profitability, cash flow, segment breakdowns, or historical trends—just let me know!

Research Output: Here is the financial snapshot for **TechCorp** based on the latest available data:

| Metric | Value |
|--------|-------|
| **Revenue** | **$450 million** |
| **Number of Employees** | **5,000** |
| **Annual Growth Rate** | **15%** |

If you need more detailed information (e.g., profit margins, cash flow, balance sheet items, historical trends, or segment breakdowns), just let me know!
--- Researcher (with tool access)----
Here is the financial snapshot for **TechCorp** based on the latest available data:

| Metric | Value |
|--------|-------|
| **Revenue** | **$450 mi

In [12]:
class ResearchTeam:
    """A complete, reusable multi-agent system"""
    def __init__(self):
        self.researcher = Agent("Researcher",
            "You are a thorough researcher. Provide factual, well-organized "
            "findings on the given topic, citing specific details.",
            temperature=0.3)
        self.critic = Agent("Critic",
            "You are a critical reviewer. Given research findings, identify "
            "any gaps, unsupported claims, or areas needing more evidence. "
            "Be constructive but rigorous.",
            temperature=0.4)
        self.writer = Agent("Writer",
            "You are a skilled writer. Given research findings AND a critic's "
            "feedback, write a balanced, well-supported final summary that "
            "addresses the critic's concerns.",
            temperature=0.5)

    def research(self, topic):
        print(f"Researching: {topic}\n")

        findings = self.researcher.run(f"Research: {topic}")
        print(f"[Researcher]: {findings[:200]}...\n")

        critique = self.critic.run(f"Review these findings:\n{findings}")
        print(f"[Critic]: {critique[:200]}...\n")

        final = self.writer.run(
            f"Findings:\n{findings}\n\nCritic's feedback:\n{critique}\n\n"
            f"Write a final, improved summary addressing the feedback."
        )
        print(f"[Writer - FINAL]: {final}")
        return final

team = ResearchTeam()
result = team.research("The benefits and risks of AI in healthcare diagnostics")

Researching: The benefits and risks of AI in healthcare diagnostics

[Researcher]: **Research Report: Benefits and Risks of AI in Healthcare Diagnostics**  
*Prepared: 17 August 2026*  

---

## 1. Executive...

[Critic]: ...

[Writer - FINAL]: 


####Sequential vs Hierarchial: Two Ways to Coordinate Agents

In [14]:
#Reuse the EXACT Agent class from cell 2 -- nothing new here
researcher = Agent("Researcher", "You are a market researcher. Provide factual findings.")
analyst    = Agent("Analyst", "You are a data analyst. Identify 3 key insights from findings.")
writer     = Agent("Writer", "You are a content writer. Write a short executive summary.")

#WRAP each agent as a callable Python function -- exactly Day 1's tool pattern
def call_researcher(topic):
  return {"output": researcher.run(f"Research: {topic}")}

def call_analyst(findings):
  return {"output": analyst.run(f"Analyst: {findings}")}

def call_writer(insights):
  return {"output": writer.run(f"Writer a summary of: {insights}")}


#Describe them as TOOLS -- identical structure to every tools list this week
orchestrator_tools = [
    {
        "type": "function",
        "function": {
            "name": "call_researcher",
            "description": "Research a topic and gather factual findings",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {"type": "string"}
                },
                "required": ["topic"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "call_analyst",
            "description": "Analyze research findings and extract key insights",
            "parameters": {
                "type": "object",
                "properties": {
                    "findings": {"type": "string"}
                },
                "required": ["findings"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "call_writer",
            "description": "Write a polished summary from analytical insights",
            "parameters": {
                "type": "object",
                "properties": {
                    "insights": {"type": "string"}
                },
                "required": ["insights"]
            }
        }
    }
]

agent_functions = {
    "call_researcher": call_researcher,
    "call_analyst"   : call_analyst,
    "call_writer"    : call_writer
}


#THE ORCHESTRATOR -- literally Day 1's run_agent(), unchanged
def run_orchestrator(user_request, max_steps=6, max_history_tokens_approx=4000):
  messages = [{"role": "user", "content": user_request}]
  for steps in range(max_steps):
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b", messages=messages,
        tools=orchestrator_tools, tool_choice="auto"
    )
    message = response.choices[0].message
    if not message.tool_calls:
      return message.content
    messages.append(message)
    for tool_call in message.tool_calls:
      fname = tool_call.function.name
      args = json.loads(tool_call.function.arguments)
      result = agent_functions[fname](**args)

      #Truncate large tool outputs before feeding them back in
      result_str = json.dumps(result)
      if len(result_str) > 800:
        result_str = result_str[:800] + ". . . [truncated]"

      print(f"  [Orchestrator delegated to]: {fname}")
      messages.append({"role": "tool", "tool_call_id": tool_call.id,
                       "content": result_str})
  return "Max steps reached."


#Test it
result = run_orchestrator("I need a well-researched summary about electric vehicle adoption trends")
print(f"\nFinal: {result}")

  [Orchestrator delegated to]: call_researcher
  [Orchestrator delegated to]: call_researcher
  [Orchestrator delegated to]: call_researcher

Final: **Electric‑Vehicle (EV) Adoption Trends – 2020 – 2024 (Key Insights & Outlook)**  

---

## 1. Global Market Size & Growth  

| Year | Total Passenger‑Vehicle EV Stock* | New EV Registrations (2023) | Global EV‑Sales Share of New Cars |
|------|-----------------------------------|----------------------------|-----------------------------------|
| 2020 | ~7.2 million | 2.1 million | 4.6 % |
| 2021 | ~10.5 million | 3.1 million | 6.6 % |
| 2022 | ~13.9 million | 4.3 million | 9.1 % |
| 2023 | ~16.9 million | **10.5 million** | **14.0 %** |
| 2024 (proj.) | ~20 million | 11‑12 million | 16‑18 % |

\*Passenger‑vehicle stock includes Battery‑Electric Vehicles (BEVs) and Plug‑in Hybrid Electric Vehicles (PHEVs).  

**Sources:** International Energy Agency (IEA) *Global EV Outlook 2024*; EV‑Volumes database; BloombergNEF (BNEF) *Electric Vehicle 